# Neural Network Fundamentals

**Chapter 7: Predicting Soccer Outcomes with Deep Learning**

## Learning Objectives

- Understand what neural networks are and how they work
- Learn about neurons, weights, and biases
- Explore activation functions (ReLU, Sigmoid, Tanh)
- Understand forward propagation
- Learn about loss functions and their role in training
- Grasp the basics of backpropagation and gradient descent

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## The Intuition: A Team of Players

Imagine a soccer team on the pitch. Each player has a role:
- **Defenders** block attacks
- **Midfielders** control the flow
- **Strikers** create scoring opportunities

Together, they process information (passes, positioning, pressing intensity) and turn it into goals.

A **neural network** works similarly. It's a team of interconnected "players" called **neurons**, organized into **layers**. These neurons work together to transform input data (match statistics) into predictions (goals scored, match outcomes).

### Three Types of Layers

1. **Input Layer**: Receives raw data (e.g., soccer statistics), with each neuron representing a specific feature
2. **Hidden Layers**: Process data and identify complex patterns. A network with 2+ hidden layers is called **deep learning**
3. **Output Layer**: Generates the network's predictions (e.g., match results)

## Neurons: The Building Blocks

Just like formations are made up of individual players, layers in neural networks are made up of **neurons**.

### Weights and Biases

The "knowledge" of a neural network is quantified in **weights**. Think of them as the strength of connections between neurons.

**Example**: Teams that take many shots on target typically score more goals. In a neural network, the connection (weight) between a neuron representing "shots on target" and a neuron representing "goals scored" would become strong as the model learns this relationship.

Each neuron also has a **bias** - a personal tendency that shifts its decision-making independent of inputs. Think of it as a player's natural inclination to attack or defend even before receiving the ball.

In [ ]:
# Simple neuron computation
def neuron_output(inputs, weights, bias):
    """
    Calculate the output of a single neuron.
    
    Parameters:
    - inputs: array of input values
    - weights: array of weights (same length as inputs)
    - bias: single bias value
    
    Returns:
    - weighted sum + bias
    """
    return np.dot(inputs, weights) + bias

# Example: Predicting if a team will score based on shots and possession
inputs = np.array([15, 60])  # [shots, possession%]
weights = np.array([0.3, 0.1])  # learned weights
bias = -2.0  # learned bias

output = neuron_output(inputs, weights, bias)
print(f"Inputs: {inputs}")
print(f"Weights: {weights}")
print(f"Bias: {bias}")
print(f"Neuron output (before activation): {output:.3f}")

## Activation Functions: Deciding When to Pass

Weights and biases alone aren't enough. We need to decide when information flows through the network and when it doesn't.

Think of a sloppy pass in soccer: if it's bad enough, the home team intercepts; if it's precise, the ball goes through. That "threshold effect" is what **activation functions** model.

### The Three Most Common Activation Functions

1. **ReLU (Rectified Linear Unit)**
2. **Sigmoid**
3. **Hyperbolic Tangent (tanh)**

### 1. ReLU (Rectified Linear Unit)

The ReLU function is simple: give an input below zero, and it outputs zero. Above zero, it equals the input value.

$$f(x) = \max(0, x)$$

**Soccer analogy**: Think of it like goal counts - there's no such thing as negative goals, but once the team starts scoring, every additional goal adds up.

**Advantages**: Makes networks efficient (lots of "zeros" where nothing happens) and flexible (many ReLUs stacked together create complex patterns)

**Drawback**: If too many inputs fall below zero, parts of the network "die" and stop learning

In [ ]:
# ReLU activation function
def relu(x):
    return np.maximum(0, x)

# Visualize ReLU
x = np.linspace(-3, 3, 100)
y_relu = relu(x)

plt.figure(figsize=(10, 5))
plt.plot(x, y_relu, linewidth=2, label='ReLU')
plt.axhline(y=0, color='k', linestyle='--', alpha=0.3)
plt.axvline(x=0, color='k', linestyle='--', alpha=0.3)
plt.xlabel('Input (x)', fontsize=12)
plt.ylabel('Output f(x)', fontsize=12)
plt.title('ReLU Activation Function', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

# Example
test_values = np.array([-2, -1, 0, 1, 2])
print(f"Input values: {test_values}")
print(f"ReLU outputs: {relu(test_values)}")

### 2. Sigmoid Function

The sigmoid function provides smooth, probabilistic outputs between 0 and 1.

$$y = \frac{1}{1 + e^{-x}}$$

**Soccer analogy**: Perfect for outputs we want to read as probabilities - for example, "What's the chance the home team wins?"

**Advantages**: Outputs are always between 0 and 1, making them interpretable as probabilities

**Drawback**: When values get too extreme, the curve flattens and learning slows (the "vanishing gradient problem")

In [ ]:
# Sigmoid activation function
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Visualize Sigmoid
y_sigmoid = sigmoid(x)

plt.figure(figsize=(10, 5))
plt.plot(x, y_sigmoid, linewidth=2, label='Sigmoid', color='orange')
plt.axhline(y=0.5, color='r', linestyle='--', alpha=0.3, label='y=0.5')
plt.axvline(x=0, color='k', linestyle='--', alpha=0.3)
plt.xlabel('Input (x)', fontsize=12)
plt.ylabel('Output f(x)', fontsize=12)
plt.title('Sigmoid Activation Function', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim(-0.1, 1.1)
plt.show()

# Example
print(f"Input values: {test_values}")
print(f"Sigmoid outputs: {sigmoid(test_values)}")
print(f"\nNote: All outputs are between 0 and 1 (probabilities)")

### 3. Hyperbolic Tangent (tanh)

Tanh is similar to sigmoid but ranges from -1 to 1, so it's centered on zero.

$$\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

**Advantages**: Can sometimes train faster than sigmoid

**Drawback**: Like sigmoid, it also suffers from vanishing gradients

We won't use it much in this chapter, but it's worth knowing as another option in the toolbox.

In [ ]:
# Tanh activation function
y_tanh = np.tanh(x)

# Visualize all three together
plt.figure(figsize=(12, 6))
plt.plot(x, y_relu, linewidth=2, label='ReLU')
plt.plot(x, y_sigmoid, linewidth=2, label='Sigmoid')
plt.plot(x, y_tanh, linewidth=2, label='Tanh')
plt.axhline(y=0, color='k', linestyle='--', alpha=0.3)
plt.axvline(x=0, color='k', linestyle='--', alpha=0.3)
plt.xlabel('Input (x)', fontsize=12)
plt.ylabel('Output f(x)', fontsize=12)
plt.title('Comparison of Activation Functions', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

## Forward Propagation: Moving the Ball Forward

Forward propagation is the process of moving data through a network to generate predictions.

Think of it like a team moving the ball up the field:
- **Passes (weights)** and **positioning (biases)** decide where the play develops
- **Activations** decide whether the ball actually goes through

Here's a minimal two-layer example:

In [ ]:
def forward_propagation(X, weights, biases):
    """
    Perform forward propagation through a two-layer neural network.
    
    Parameters:
    - X: input features
    - weights: list of weight matrices [weights_layer1, weights_layer2]
    - biases: list of bias vectors [biases_layer1, biases_layer2]
    
    Returns:
    - Output probabilities
    """
    # Compute the linear transformation for the hidden layer: Z = XW + b
    z1 = np.dot(X, weights[0]) + biases[0]
    
    # Apply the ReLU activation function to introduce non-linearity
    a1 = relu(z1)
    
    # Compute the linear transformation for the output layer: Z = AW + b
    z2 = np.dot(a1, weights[1]) + biases[1]
    
    # Apply the Sigmoid activation function to produce output probabilities
    a2 = sigmoid(z2)
    
    return a2

# Example: Simple network
np.random.seed(42)
X_example = np.array([[15, 60]])  # [shots, possession%]

# Initialize random weights and biases
weights = [
    np.random.randn(2, 4) * 0.1,  # Input to hidden (2 inputs, 4 hidden neurons)
    np.random.randn(4, 1) * 0.1   # Hidden to output (4 hidden, 1 output)
]
biases = [
    np.zeros(4),  # Hidden layer bias
    np.zeros(1)   # Output layer bias
]

# Forward pass
output = forward_propagation(X_example, weights, biases)
print(f"Input: {X_example}")
print(f"Predicted probability of home win: {output[0][0]:.3f}")

## Loss Functions: Keeping Score

How do we judge whether a model is "playing well"? In soccer, the scoreboard tells us if a team's strategy is working. In machine learning, that scoreboard is the **loss function**: a number that measures how far off the model's predictions are from reality.

A **lower loss** means our predictions are closer to the truth. During training, the network constantly updates its weights to reduce this loss.

### Common Loss Functions for Soccer Tasks

1. **Binary Cross-Entropy (BCE)**: For yes/no tasks like "Will the home team win?" (1 for win, 0 for not win)
2. **Categorical Cross-Entropy**: For multi-class outcomes like predicting win/draw/loss
3. **Mean Squared Error (MSE)**: For continuous values, such as predicting goals scored
4. **Poisson Regression Loss**: For count data, e.g., predicting how many goals a team scores

In [ ]:
# Binary Cross-Entropy Loss
def binary_cross_entropy(y_true, y_pred):
    """
    Calculate binary cross-entropy loss.
    
    Parameters:
    - y_true: actual labels (0 or 1)
    - y_pred: predicted probabilities (between 0 and 1)
    
    Returns:
    - loss value
    """
    epsilon = 1e-15  # Small value to avoid log(0)
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Example
y_true = np.array([1, 0, 1, 1, 0])  # Actual outcomes
y_pred_good = np.array([0.9, 0.1, 0.85, 0.95, 0.05])  # Good predictions
y_pred_bad = np.array([0.5, 0.5, 0.5, 0.5, 0.5])  # Random predictions

loss_good = binary_cross_entropy(y_true, y_pred_good)
loss_bad = binary_cross_entropy(y_true, y_pred_bad)

print(f"True labels: {y_true}")
print(f"\nGood predictions: {y_pred_good}")
print(f"Loss (good model): {loss_good:.4f}")
print(f"\nBad predictions: {y_pred_bad}")
print(f"Loss (bad model): {loss_bad:.4f}")
print(f"\nLower loss = better predictions!")

In [ ]:
# Mean Squared Error (for regression tasks)
def mean_squared_error(y_true, y_pred):
    """
    Calculate mean squared error.
    
    Parameters:
    - y_true: actual values
    - y_pred: predicted values
    
    Returns:
    - MSE value
    """
    return np.mean((y_true - y_pred) ** 2)

# Example: Predicting goals scored
y_true_goals = np.array([2, 1, 3, 0, 2])  # Actual goals
y_pred_goals_good = np.array([2.1, 0.9, 2.8, 0.2, 1.9])  # Good predictions
y_pred_goals_bad = np.array([0, 0, 1, 2, 3])  # Bad predictions

mse_good = mean_squared_error(y_true_goals, y_pred_goals_good)
mse_bad = mean_squared_error(y_true_goals, y_pred_goals_bad)

print(f"True goals: {y_true_goals}")
print(f"\nGood predictions: {y_pred_goals_good}")
print(f"MSE (good model): {mse_good:.4f}")
print(f"\nBad predictions: {y_pred_goals_bad}")
print(f"MSE (bad model): {mse_bad:.4f}")

## How a Neural Network Learns

Before we can train a model to make good predictions, we need to understand how it learns from mistakes.

In soccer terms: it's not enough to just play forward - the team has to review what went wrong after a match, adjust its strategy, and improve for next time.

### Backpropagation: Reviewing the Game Tape

**Backpropagation** is the method a neural network uses to learn from errors. Imagine replaying a failed attack in slow motion: the coach rewinds the video from the missed shot all the way back to the first pass, pointing out where each player could adjust.

Backpropagation takes the prediction error at the output layer and works backwards through the network, showing each weight how much it contributed to the mistake.

The key output is the **gradient**: a measure of how much changing a specific weight would change the loss.

### Gradient Descent: Making Adjustments

Backprop tells each weight how it contributed to an error; **gradient descent** is how the network uses that feedback to improve.

The idea is simple: adjust the weights little by little in the direction that makes the loss smaller.

Think of a coach fine-tuning tactics after a match. If the team pressed too aggressively and got caught on the counter, next time they press a little less. Each game is an "iteration," nudging strategy closer to what works.

In [ ]:
# Simplified gradient descent visualization
def loss_function(w):
    """Simple quadratic loss function for visualization"""
    return (w - 3) ** 2

# Gradient descent
learning_rate = 0.1
w = 0  # Starting weight
history = [w]

for i in range(20):
    # Calculate gradient (derivative of loss)
    gradient = 2 * (w - 3)
    
    # Update weight
    w = w - learning_rate * gradient
    history.append(w)

# Visualize
w_range = np.linspace(-1, 6, 100)
loss_range = loss_function(w_range)

plt.figure(figsize=(12, 5))
plt.plot(w_range, loss_range, linewidth=2, label='Loss Function')
plt.scatter(history, [loss_function(w) for w in history], 
            c=range(len(history)), cmap='viridis', s=50, zorder=5)
plt.xlabel('Weight Value', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Gradient Descent: Finding the Optimal Weight', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.colorbar(label='Iteration')
plt.show()

print(f"Starting weight: {history[0]:.3f}")
print(f"Final weight: {history[-1]:.3f}")
print(f"Optimal weight: 3.000")
print(f"\nThe algorithm successfully found the minimum!")

### Three Styles of Gradient Descent

1. **Batch Gradient Descent**: Uses the entire dataset to make one update. Very stable, but slow if the dataset is huge.

2. **Stochastic Gradient Descent (SGD)**: Updates after each single sample. Faster to react, but the learning path can be noisy.

3. **Mini-Batch Gradient Descent**: A commonly-used balance - updates after a handful of samples at a time, combining stability with speed.

## Summary

In this notebook, you learned:

✅ **Neural networks** are composed of layers of neurons that process information

✅ **Weights** represent the strength of connections between neurons

✅ **Biases** allow neurons to activate more flexibly

✅ **Activation functions** (ReLU, Sigmoid, Tanh) determine when information flows through the network

✅ **Forward propagation** is the process of moving data through the network to make predictions

✅ **Loss functions** measure how far predictions are from reality

✅ **Backpropagation** calculates how each weight contributed to the error

✅ **Gradient descent** uses those gradients to update weights and improve the model

### Next Steps

In the next notebook, we'll put these concepts into practice by building our first neural network in PyTorch to predict home team wins!

## Practice Exercises

1. **Experiment with activation functions**: Create a neuron with different activation functions and observe how the output changes with different inputs.

2. **Build a 3-layer network**: Extend the `forward_propagation` function to include an additional hidden layer.

3. **Loss comparison**: Create predictions for a classification task and calculate both BCE and MSE. Which one is more appropriate and why?

4. **Gradient descent tuning**: Modify the learning rate in the gradient descent example. What happens with very large (e.g., 0.9) or very small (e.g., 0.01) learning rates?

5. **Soccer application**: Think of a soccer prediction task you'd like to solve. What would be your input features? What activation function would you use in the output layer? What loss function would be appropriate?